# Sample images from LILA BC

1. Get image URLs and labels from LILA BC
2. Sample n images for each class
3. Train val test split
4. Save CSV with sampled image URLs to Drive

## Setup

In [1]:
import os
import sys
import yaml
import random
import subprocess
import numpy as np
import pandas as pd
import polars as pl
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# add folder to sys.path so that utilities can be imported
project_dir = 'drive/MyDrive/TeraiNet'
sys.path.append(project_dir)

In [4]:
from utilities import load_config, sample_n_images_per_species, add_subset_column, check_location_split

In [5]:
# load config and set variables and parameters
config_path = os.path.join(project_dir, 'config.yaml')
config = load_config(config_path)

scripts_dir = os.path.join(project_dir, config['sampling_downloading']['scripts_dir'])

samples_dir = os.path.join(project_dir, config['sampling_downloading']['samples_dir'])

train_ratio = config['training']['train_ratio']

In [6]:
# for a fresh start, remove samples dir
remove_samples_dir = True
if remove_samples_dir:
  !rm -rf "$samples_dir"
  !mkdir -p "$samples_dir"

## Download image URLs and labels from LILA BC

In [7]:
!wget -O lila_image_urls_and_labels.csv.zip -nc "https://lila.science/public/lila_image_urls_and_labels.csv.zip"

File ‘lila_image_urls_and_labels.csv.zip’ already there; not retrieving.


In [8]:
![ -f lila_image_urls_and_labels.csv ] || unzip lila_image_urls_and_labels.csv.zip

In [9]:
urls_and_labels = 'lila_image_urls_and_labels.csv'

## Inspect species counts

Check taxonomy mapping to find relevant species:
https://lila.science/public/lila-taxonomy-mapping_release.csv

In [10]:
columns = [
    'url_gcp',
    'image_id',
    'sequence_id',
    'location_id',
    'frame_num',
    'datetime',
    'common_name'
]

schema_overrides = {
    'url_gcp': pl.Utf8(),
    'image_id': pl.Utf8(),
    'sequence_id': pl.Utf8(),
    'location_id': pl.Utf8(),
    'frame_num': pl.Int32(),
    'datetime': pl.Utf8(),
    'common_name': pl.Utf8()
}

lila_image_urls_and_labels_df = pl.read_csv(
    'lila_image_urls_and_labels.csv',
    columns=columns,
    schema_overrides=schema_overrides
)
lila_image_urls_and_labels_df.shape

(23723546, 7)

In [11]:
lila_image_urls_and_labels_df.head()

url_gcp,image_id,sequence_id,location_id,frame_num,common_name,datetime
str,str,str,str,i32,str,str
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5968c0f…","""Caltech Camera Traps : 6f2160e…","""Caltech Camera Traps : 26""",1,null,"""10-04-2013 13:31:53"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5a0b016…","""Caltech Camera Traps : 6f27ed6…","""Caltech Camera Traps : 26""",1,"""deer""","""11-04-2013 18:37:07"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 59b93af…","""Caltech Camera Traps : 6f04895…","""Caltech Camera Traps : 38""",2,"""cat""","""05-09-2012 07:33:45"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 59641f5…","""Caltech Camera Traps : 6f0385b…","""Caltech Camera Traps : 38""",2,"""virginia opossum""","""03-29-2012 02:34:13"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5a1e530…","""Caltech Camera Traps : 6f0a3cc…","""Caltech Camera Traps : 33""",2,null,"""05-08-2012 19:23:36"""


In [12]:
# TODO: figure out why common_name is empty in half the rows and what this implies
lila_image_urls_and_labels_df.filter(pl.col('common_name').is_null()).shape

(11843789, 7)

In [13]:
species_list = [
    'tiger',
    'leopard',
    'asian black bear', 'american black bear', # not enough images of asian black bear alone
    'dhole', 'black-backed jackal', 'gray fox', 'leopard cat', 'mainland leopard cat', 'marbled cat', 'asian golden cat', # other carnivores (including substitutes, i. e. black-backed jackal and gray fox)
    'deer',
    'wild boar',
    'african buffalo', 'cape buffalo', # substitute for gaur
    'white rhinoceros', # substitute for indian rhino
    'asian elephant', 'african bush elephant', # not enough images of asian elephant alone
    'bird'
]

In [14]:
lila_image_urls_and_labels_df = lila_image_urls_and_labels_df.filter(pl.col('common_name').is_in(species_list))
species_stats = lila_image_urls_and_labels_df.group_by('common_name').agg(
    pl.len().alias('total_count'),
    pl.n_unique('location_id').alias('unique_location_count')
)
pl.Config.set_tbl_rows(species_stats.shape[0])
species_stats.sort('total_count')

common_name,total_count,unique_location_count
str,u32,u32
"""dhole""",185,36
"""mainland leopard cat""",246,95
"""leopard cat""",266,31
"""marbled cat""",271,86
"""tiger""",321,109
"""asian elephant""",325,78
"""asian golden cat""",353,143
"""asian black bear""",1221,54
"""leopard""",2991,559


## Get image URLs from LILA BC

The goal is to get about 3000 images per class (+100 buffer). Leopard images availability sets this limit because we want classes to be balanced. For tiger images, which are even scarcer, there fortunately is another additional source (Amur tiger re-identification challenge)

### Tiger

In [15]:
class_name = 'tiger'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['tiger']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 321


In [16]:
# use all tiger images because we don't have many
species_samples_dict = {
    'tiger': all_image_urls.shape[0]
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
# put all lila bc tiger images into separate test set to test how well model trained on amur tiger images generalizes
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number'),
        pl.lit('test2').alias('subset')
    )
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"

ℹ️ Species distribution:
shape: (1, 2)
┌─────────────┬───────┐
│ common_name ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ tiger       ┆ 321   │
└─────────────┴───────┘
ℹ️ Total sampled rows: 321


### Leopard

In [17]:
class_name = 'leopard'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['leopard']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 2991


In [18]:
# use all leopard images because because we don't have many
species_samples_dict = {
    'leopard': 2991
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 368   │
│ train  ┆ 2262  │
│ val    ┆ 361   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (1, 2)
┌─────────────┬───────┐
│ common_name ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ leopard     ┆ 2991  │
└─────────────┴───────┘
ℹ️ Total sampled rows: 2991
✅ No violations found: each location_id appears in only one subset.


### Black bear

Include `american black bear` because there are not enough camera trap images of Asian black bears.

In [19]:
class_name = 'black_bear'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = [
    'asian black bear',
    'american black bear'
]
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 34075


In [20]:
species_samples_dict = {
    'asian black bear': 1221,
    'american black bear': 1879
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 491   │
│ train  ┆ 2233  │
│ val    ┆ 376   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (2, 2)
┌─────────────────────┬───────┐
│ common_name         ┆ count │
│ ---                 ┆ ---   │
│ str                 ┆ u32   │
╞═════════════════════╪═══════╡
│ american black bear ┆ 1879  │
│ asian black bear    ┆ 1221  │
└─────────────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Other carnivores

Include `dhole,black-backed jackal,gray fox,leopard cat,mainland leopard cat,marbled cat,asian golden cat` to cover a wide range of other carnivores in the Terai ecosystem.

In [21]:
class_name = 'other_carnivores'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = [
    'dhole',
    'black-backed jackal',
    'gray fox',
    'leopard cat',
    'mainland leopard cat',
    'marbled cat',
    'asian golden cat'
]
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 34402


In [22]:
species_samples_dict = {
    'dhole': 185,
    'black-backed jackal': 890,
    'gray fox': 890,
    'leopard cat': 266,
    'mainland leopard cat': 246,
    'marbled cat': 271,
    'asian golden cat': 353
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 374   │
│ train  ┆ 2354  │
│ val    ┆ 373   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (7, 2)
┌──────────────────────┬───────┐
│ common_name          ┆ count │
│ ---                  ┆ ---   │
│ str                  ┆ u32   │
╞══════════════════════╪═══════╡
│ marbled cat          ┆ 271   │
│ gray fox             ┆ 890   │
│ dhole                ┆ 185   │
│ black-backed jackal  ┆ 890   │
│ mainland leopard cat ┆ 246   │
│ asian golden cat     ┆ 353   │
│ leopard cat          ┆ 266   │
└──────────────────────┴───────┘
ℹ️ Total sampled rows: 3101
✅ No violations found: each location_id appears in only one subset.


### Deer

In [23]:
class_name = 'deer'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['deer']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 360489


In [24]:
species_samples_dict = {
    'deer': 3100
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 398   │
│ train  ┆ 2328  │
│ val    ┆ 374   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (1, 2)
┌─────────────┬───────┐
│ common_name ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ deer        ┆ 3100  │
└─────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Wild boar

In [25]:
class_name = 'wild_boar'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['wild boar']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 142701


In [26]:
species_samples_dict = {
    'wild boar': 3100
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 429   │
│ train  ┆ 2273  │
│ val    ┆ 398   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (1, 2)
┌─────────────┬───────┐
│ common_name ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ wild boar   ┆ 3100  │
└─────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Buffalo

Use `african buffalo,cape buffalo` because camera trap images of gaur are unavailable.

In [27]:
class_name = 'buffalo'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = [
    'african buffalo',
    'cape buffalo'
]
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 94114


In [28]:
species_samples_dict = {
    'african buffalo': 1550,
    'cape buffalo': 1550
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 420   │
│ train  ┆ 2266  │
│ val    ┆ 414   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (2, 2)
┌─────────────────┬───────┐
│ common_name     ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ african buffalo ┆ 1550  │
│ cape buffalo    ┆ 1550  │
└─────────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Rhino

Use `white rhinoceros` because camera trap images of Indian rhinoceros are unavailable.

In [29]:
class_name = 'rhino'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['white rhinoceros']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 7307


In [30]:
species_samples_dict = {
    'white rhinoceros': 3100
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 373   │
│ train  ┆ 2142  │
│ val    ┆ 585   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (1, 2)
┌──────────────────┬───────┐
│ common_name      ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ white rhinoceros ┆ 3100  │
└──────────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Elephant

Include `african bush elephant` because there are not enough camera trap images of Asian elephants.

In [31]:
class_name = 'elephant'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = [
    'asian elephant',
    'african bush elephant'
]
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 187445


In [32]:
species_samples_dict = {
    'asian elephant': 325,
    'african bush elephant': 2775
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 378   │
│ train  ┆ 2349  │
│ val    ┆ 373   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (2, 2)
┌───────────────────────┬───────┐
│ common_name           ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ asian elephant        ┆ 325   │
│ african bush elephant ┆ 2775  │
└───────────────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.


### Bird

In [33]:
class_name = 'bird'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = ['bird']
all_image_urls = lila_image_urls_and_labels_df.filter(pl.col(column_to_filter).is_in(values_to_filter))
print(f'All rows: {all_image_urls.shape[0]}')

All rows: 297686


In [34]:
species_samples_dict = {
    'bird': 3100
}

sampled_image_urls = sample_n_images_per_species(all_image_urls, species_samples_dict, column_to_filter)
sampled_image_urls = sampled_image_urls.with_columns(
        pl.lit(class_number).alias('class_number')
    )
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=',')
print(f'ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}')
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

ℹ️ Subset distribution:
shape: (3, 2)
┌────────┬───────┐
│ subset ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ test   ┆ 377   │
│ train  ┆ 2349  │
│ val    ┆ 374   │
└────────┴───────┘
ℹ️ Species distribution:
shape: (1, 2)
┌─────────────┬───────┐
│ common_name ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ bird        ┆ 3100  │
└─────────────┴───────┘
ℹ️ Total sampled rows: 3100
✅ No violations found: each location_id appears in only one subset.
